# DBSCAN absolute-radius attribution controls

Two follow-up controls for the clustering feature screen. Use the radii from the two feature candidates that tie sample F1 and increase macro F1, but remove the added features. All original labels and classifier settings remain unchanged. These are adaptive checks, separate from the initial 28-setting grid.

In [1]:
def run_absolute_radius_controls():
    from pathlib import Path
    import json,hashlib,warnings
    import joblib,numpy as np,pandas as pd
    from sklearn.base import clone
    from sklearn.cluster import DBSCAN
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import f1_score
    from threadpoolctl import threadpool_limits
    cwd=Path.cwd()
    data=next(p for p in [cwd,cwd/'part1',cwd/'AML-Midterm/part1'] if (p/'data/part1_train.csv').is_file() or (p/'part1_train.csv').is_file()).resolve()
    raw=data if (data/'part1_train.csv').is_file() else data/'data'
    main=data/'experiments/clustering_feature_screen';out=main/'absolute_radius_controls'
    for folder in ['datasets','labeling_models','classifier_models','validation_predictions']:(out/folder).mkdir(parents=True,exist_ok=True)
    reasons=['Safety','Funding','Efficacy','Recruitment','Other'];features=[f'emb_{i}' for i in range(150)]
    train=pd.read_csv(raw/'part1_train.csv');valid=pd.read_csv(raw/'part1_val.csv')
    original=train[reasons].to_numpy(dtype=int);truth=valid[reasons].to_numpy(dtype=int)
    known=train[reasons].isin([0,1]).all(axis=1).to_numpy();assert known.sum()==157
    source=data/'experiments/additional_methods'
    seed=joblib.load(source/'classifier_models/dbscan_m5_q25__logistic_C1000.joblib')
    cluster_scaler=joblib.load(source/'labeling_models/dbscan_m5_q25.joblib')['scaler']
    xc=cluster_scaler.transform(train[features].to_numpy());xs=seed['scaler'].transform(train[features].to_numpy())
    vs=seed['scaler'].transform(valid[features].to_numpy())
    rule=json.loads((main/'settings.json').read_text())['label_rule'];rows=[]
    protected={str(p.relative_to(data)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [raw/'part1_train.csv',raw/'part1_val.csv',main/'comparison.csv',main/'settings.json']}
    for source_name in ['dbscan_diseases_scale0p25_q50','dbscan_criteria_scale0p25_q50']:
        expanded=joblib.load(main/'labeling_models'/f'{source_name}.joblib');eps=float(expanded['clustering'].eps)
        name=f'embeddings_at_{source_name}_radius'
        with threadpool_limits(limits=1):density=DBSCAN(eps=eps,min_samples=5,metric='euclidean',algorithm='brute',n_jobs=1).fit(xc)
        ids=density.labels_;clusters=np.unique(ids[ids>=0])
        y=np.full_like(original,-100);w=np.zeros(original.shape);y[known]=original[known];w[known]=1.0
        if len(clusters):
            membership=np.column_stack([ids==c for c in clusters]).astype(float)
            support=membership[known].sum(axis=0)
            rates=(membership[known].T@original[known]+rule['prior_strength']*original[known].mean(axis=0))/(support[:,None]+rule['prior_strength'])
            probability=membership@rates;support=membership@support
            eligible=(~known)&(ids>=0)&(support>=rule['minimum_support'])
            for j in range(5):
                positive=eligible&(probability[:,j]>=rule['positive_threshold']);negative=eligible&(probability[:,j]<=rule['negative_threshold'])
                np_,nn=int(positive.sum()),int(negative.sum());kp=int(original[known,j].sum());kn=int(known.sum()-kp)
                if not np_ or not nn:continue
                fraction=min(rule['maximum_class_fraction'],rule['maximum_generated_weight']*np_/kp,rule['maximum_generated_weight']*nn/kn)
                y[positive,j]=1;y[negative,j]=0;w[positive,j]=fraction*kp/np_;w[negative,j]=fraction*kn/nn
        np.testing.assert_array_equal(y[known],original[known]);np.testing.assert_array_equal(w[known],np.ones((157,5)))
        assert (w[(ids<0)&~known]==0).all()
        model={**seed,'estimators':[],'dataset':name}
        with threadpool_limits(limits=1):
            for j in range(5):
                included=w[:,j]>0;estimator=clone(seed['estimators'][j])
                with warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    estimator.fit(xs[included],y[included,j],sample_weight=w[included,j]*157/w[included,j].sum())
                model['estimators'].append(estimator)
        probabilities=np.column_stack([e.predict_proba(vs)[:,list(e.classes_).index(1)] for e in model['estimators']])
        prediction=(probabilities>=0.5).astype(int);empty=prediction.sum(axis=1)==0
        indices=np.flatnonzero(empty);prediction[indices,probabilities[indices].argmax(axis=1)]=1
        sample=f1_score(truth,prediction,average='samples',zero_division=0);macro=f1_score(truth,prediction,average='macro',zero_division=0)
        expanded_prediction=pd.read_csv(main/'validation_predictions'/f'{source_name}.csv')[[f'predicted_{r}' for r in reasons]].to_numpy(dtype=int)
        row={'experiment':name,'radius_source':source_name,'eps':eps,'sample_f1':sample,'macro_f1':macro,
             'different_trial_predictions_vs_expanded':int((prediction!=expanded_prediction).any(axis=1).sum()),
             'generated_trials':int(((w>0).any(axis=1)&~known).sum()),'clusters':len(clusters),'noise_trials':int((ids<0).sum())}
        rows.append(row)
        frame=train[['ID']+features].copy()
        for j,r in enumerate(reasons):frame[r]=y[:,j];frame[f'weight_{r}']=w[:,j]
        frame.to_csv(out/'datasets'/f'{name}.csv',index=False)
        predicted_frame=valid[['ID']+reasons].rename(columns={r:f'actual_{r}' for r in reasons}).copy()
        for j,r in enumerate(reasons):predicted_frame[f'probability_{r}']=probabilities[:,j];predicted_frame[f'predicted_{r}']=prediction[:,j]
        predicted_frame['used_fallback']=empty;predicted_frame.to_csv(out/'validation_predictions'/f'{name}.csv',index=False)
        joblib.dump(model,out/'classifier_models'/f'{name}.joblib',compress=3)
        joblib.dump({'clustering':density,'scaler':cluster_scaler,'context':None,'label_rule':rule},out/'labeling_models'/f'{name}.joblib',compress=3)
        loaded=joblib.load(out/'classifier_models'/f'{name}.joblib')
        np.testing.assert_allclose(np.column_stack([e.predict_proba(vs)[:,list(e.classes_).index(1)] for e in loaded['estimators']]),probabilities,rtol=1e-12,atol=1e-12)
        print(f'{source_name}: embeddings-only at eps={eps:.12g}, sample F1={sample:.6f}, macro F1={macro:.6f}, changed trials vs expanded={row["different_trial_predictions_vs_expanded"]}')
    result=pd.DataFrame(rows);result.to_csv(out/'comparison.csv',index=False)
    for rel,h in protected.items():assert hashlib.sha256((data/rel).read_bytes()).hexdigest()==h
    settings={'purpose':'Adaptive attribution check after the initial 28-setting screen; use the same absolute radii as the two primary-F1 ties with higher macro F1, but remove added features.',
              'configurations':2,'classifier_C':1000,'classifier_features':features,'cutoff':0.5,'fallback':True,
              'label_rule':rule,'extra_features_in_clustering':False,'no_refitting_with_validation_labels':True,
              'protected_input_sha256':protected,'new_kaggle_scores':None}
    (out/'settings.json').write_text(json.dumps(settings,indent=2)+'\n')
    lines=['# Absolute-radius attribution controls','','Two adaptive controls reuse the exact absolute DBSCAN radii of the disease/criteria scale-0.25, median-neighbor-radius candidates, but cluster on the original embeddings only. This separates a radius effect from an added-feature effect. Classifier, label rules, weights, cutoff, and fallback stay fixed.','',result.to_string(index=False),'','No validation/test fitting, final validation-label refit, Kaggle submission, commit, or tag. All saved classifier reloads and protected-input checks passed.']
    (out/'SUMMARY.md').write_text('\n'.join(lines)+'\n')
    return result

absolute_radius_control_results=run_absolute_radius_controls()


dbscan_diseases_scale0p25_q50: embeddings-only at eps=10.0954243868, sample F1=0.893162, macro F1=0.804324, changed trials vs expanded=4
dbscan_criteria_scale0p25_q50: embeddings-only at eps=10.1378433725, sample F1=0.893162, macro F1=0.804324, changed trials vs expanded=4
